# 01 — LangChain 1.0 basics: `create_agent`

This notebook covers:
- Loading credentials from a `.env` file
- Setting up `ChatOpenAI` (the `langchain-openai` provider)
- Inspecting a model's `.profile` (new in LangChain 1.1)
- Building your first agent with `create_agent` (the LangChain 1.0 replacement for `AgentExecutor` / `langgraph.prebuilt.create_react_agent`)
- Reading responses via `.content_blocks`

**Prereqs**
```bash
pip install langchain langchain-openai python-dotenv
```

Create a `.env` file next to this notebook:
```
OPENAI_API_KEY=sk-...
```


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env from the current working directory

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("Environment loaded OK")


## 1. Set up the model

`ChatOpenAI` from `langchain-openai` is the standard chat-model wrapper.
Nothing about this changes in 1.0 — integration packages stayed stable;
it's the `langchain` package itself (agents, middleware) that changed.


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
llm

## 2. Inspect the model profile (new in 1.1)

Every chat model now exposes a `.profile` describing what it actually
supports (structured output, tool calling, JSON mode, etc.), sourced
from the open `models.dev` project. This lets you branch logic on
capability instead of guessing per-provider.


In [ ]:
# ============ INSPECT MODEL PROFILE ============
import json

profile = llm.profile

# `profile` is a plain dict (a ModelProfile TypedDict), so json.dumps
# renders it readably. `default=str` guards against any non-JSON-native
# values (sets, enums, types) some providers include.
print(json.dumps(profile, indent=2, default=str, sort_keys=True))


## 3. Build your first agent with `create_agent`

`create_agent` is the new, single entry point for building agents in
LangChain 1.0. It replaces the old `AgentExecutor` and
`langgraph.prebuilt.create_react_agent`. Same high-level interface,
but it's now built directly on the LangGraph runtime.


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    # Stubbed for the demo — swap in a real weather API call.
    fake_data = {"bengaluru": "28C, humid", "kolkata": "31C, sunny"}
    return fake_data.get(city.lower(), "No data for that city")


agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a concise assistant. Use tools when relevant.",
)

# `create_agent` returns a compiled LangGraph. Ending the cell with a bare
# `agent` makes Jupyter call its `_repr_mimebundle_`, which renders the graph
# via `draw_mermaid_png()` — that POSTs to the mermaid.ink web service and
# raises ValueError when it can't be reached. Inspect it locally instead:
print(type(agent).__name__)
print("Nodes:", list(agent.get_graph().nodes))
print("Edges:", [(e.source, e.target) for e in agent.get_graph().edges])

# `.draw_mermaid()` gives the diagram source with no network call; only the
# `_png` variants go out to the API.
# print(agent.get_graph().draw_mermaid())

In [ ]:
agent

## 4. Invoke the agent

In [ ]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "What's the weather in Bengaluru right now?"}]
})

# for m in result["messages"]:
#     print(f"[{m.type}] {m.content}")


for m in result["messages"]:
    print(type(m).__name__, "|", repr(m.content))
    if getattr(m, "tool_calls", None):
        print("   tool_calls:", m.tool_calls)


### Why is the second message's content empty?

The run above prints four messages, and the *first* `AIMessage` looks blank:

```
[human] What's the weather in Bengaluru right now?
[ai]
[tool] 28C, humid
[ai] The current weather in Bengaluru is 28C and humid.
```

That is not a bug — **that AI message *is* the tool call.**

| # | Message | What it carries |
|---|---|---|
| 1 | `HumanMessage` | your question |
| 2 | `AIMessage` | `content=""`, but `tool_calls=[{'name': 'get_weather', 'args': {'city': 'Bengaluru'}, 'id': 'call_...'}]` |
| 3 | `ToolMessage` | `"28C, humid"` — what `get_weather` returned |
| 4 | `AIMessage` | the final natural-language answer |

When the model decides to call a tool, it emits *only* the tool call: there is
no prose to say alongside it, so `.content` is an empty string and the real
payload lives in `.tool_calls`. Printing only `.content` (or calling
`pretty_print()` and reading just the body) therefore shows a blank line.

Two ways to see what is actually there:

- `m.tool_calls` — the normalized list of calls, as in the loop above.
- `m.content_blocks` — the 1.x version-agnostic view; that message comes back
  as `[{'type': 'tool_call', 'name': 'get_weather', 'args': {...}, 'id': '...'}]`
  rather than an empty string.

Some models (Anthropic, and OpenAI's reasoning models) *do* emit a short
"Let me check the weather..." preamble next to the call, so message 2 has both
text and a tool call. Never assume an `AIMessage` is either text or a tool
call — it can be both, or neither.


## 5. Reading `.content_blocks`

LLM APIs increasingly return **lists of content blocks** instead of a plain
string. A single assistant turn can mix text, tool calls, reasoning traces,
citations, and images — and every provider shapes that differently:
OpenAI puts tool calls in `tool_calls`, Anthropic in `content` entries of
`{"type": "tool_use"}`, Google in `functionCall` parts.

`.content_blocks` is LangChain 1.0's **standard, provider-independent view** of
a message. It always returns a list of typed dicts, each with a `"type"` key:

| Block type | Meaning |
|---|---|
| `text` | plain output text (`{"type": "text", "text": "..."}`) |
| `tool_call` | a requested tool invocation, with `name`, `args`, `id` |
| `reasoning` | the model's thinking, where the provider exposes it |
| `citation` | source attribution attached to a span of text |
| `image` / `audio` / `file` | non-text content |

`.content` stays whatever the provider gave you (a `str` for OpenAI, a `list`
for Anthropic); `.content_blocks` is the normalized projection. Branch on
`block["type"]` and your code runs unchanged when you swap models — that is
the whole point of the abstraction.

This also answers the blank-message question from a different angle: the
tool-calling `AIMessage` is not empty, it simply has no `text` block.


In [ ]:
# ============ CONTENT BLOCKS ACROSS THE WHOLE RUN ============
# The final answer is a single text block...
final_message = result["messages"][-1]
print("Raw content:", repr(final_message.content))
print("Content blocks:", final_message.content_blocks)

# ...but walk every message and the "empty" AI turn reveals its tool_call block.
print("" + "=" * 60)
for i, m in enumerate(result["messages"]):
    print(f"[{i}] {type(m).__name__}  content={m.content!r}")
    for block in getattr(m, "content_blocks", []) or []:
        kind = block.get("type")
        if kind == "text":
            print(f"      text        -> {block['text'][:70]}")
        elif kind == "tool_call":
            print(f"      tool_call   -> {block['name']}({block['args']})")
        else:
            print(f"      {kind:<11} -> {block}")


## Recap

| Old (pre-1.0) | New (1.0+) |
|---|---|
| `AgentExecutor` + `initialize_agent` | `create_agent` |
| Implicit / hidden prompt scaffolding | Explicit `system_prompt`, explicit tools |
| Plain string `.content` only | Standardized `.content_blocks` |
| No capability introspection | `.profile` on every chat model |

Next notebook: **02 — Middleware**, where we customize this same
`create_agent` loop without dropping into raw LangGraph.
